#

# WoMakers Code 🦋 | Vagas Tech

In [ ]:
print("Instalando o repositório de pacotes Microsoft...")
!wget https://packages.microsoft.com/config/ubuntu/22.04/packages-microsoft-prod.deb -O packages-microsoft-prod.deb -q
!dpkg -i packages-microsoft-prod.deb
!rm packages-microsoft-prod.deb

print("Instalando o .Net 8 SDK...por favor, aguarde.")
!apt-get update -y -q > /dev/null
!apt-get install -y dotnet-sdk-8.0 -q > /dev/null

print("\nVerificando a instalação do .NET...")
!dotnet --version

Instalando o repositório de pacotes Microsoft...
Selecting previously unselected package packages-microsoft-prod.
(Reading database ... 122797 files and directories currently installed.)
Preparing to unpack packages-microsoft-prod.deb ...
Unpacking packages-microsoft-prod (1.0-ubuntu22.04.1) ...
Setting up packages-microsoft-prod (1.0-ubuntu22.04.1) ...
Instalando o .Net 8 SDK...por favor, aguarde.
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)

Verificando a instalação do .NET...
=8.0.425


In [ ]:
print("Criando o novo projeto de console C# 'VagasTech'...")
#Cria o projeto na pasta 'VagasTechApp'
!dotnet new console -n VagasTechApp --force

print("\nInstalando a biblioteca Microsoft.Data.Sqlite para execução de SQL...")
# Adiciona o driver oficial do SQLite para C# apontando explicitamente para o projeto
!dotnet add VagasTechApp/VagasTechApp.csproj package Microsoft.Data.Sqlite -v 8.0.11

Criando o novo projeto de console C# 'VagasTech'...
=========
Welcome to .NET 8.0!
---------------------
SDK Version: 8.0.425

Telemetry
---------
The .NET tools collect usage data in order to help us improve your experience. It is collected by Microsoft and shared with the community. You can opt-out of telemetry by setting the DOTNET_CLI_TELEMETRY_OPTOUT environment variable to '1' or 'true' using your favorite shell.

Read more about .NET CLI Tools telemetry: https://aka.ms/dotnet-cli-telemetry

----------------
Installed an ASP.NET Core HTTPS development certificate.
To trust the certificate, view the instructions: https://aka.ms/dotnet-https-linux

----------------
Write your first app: https://aka.ms/dotnet-hello-world
Find out what's new: https://aka.ms/dotnet-whats-new
Explore documentation: https://aka.ms/dotnet-docs
Report issues and find source on GitHub: https://github.com/dotnet/core
Use 'dotnet --help' to see available commands or visit: https://aka.ms/dotnet-cli


In [ ]:
import sqlite3

try:
  print("Criando o arquivo fisico do banco de dados 'vagastech.db'...")
  # Abre a conexão (se o arquivo não existir, o SQLite cria um arquivo .db vazio na raiz)
  conexao = sqlite3.connect('vagastech.db')
  print("Banco 'vagastech.db' criado fisicamente na pasta de arquivos!")
  conexao.close()
except Exception as e:
  print(f"Erro ao criar banco de :{e}")

Criando o arquivo fisico do banco de dados 'vagastech.db'...
Banco 'vagastech.db' criado fisicamente na pasta de arquivos!


In [ ]:
  %%writefile VagasTechApp/MetodosCRUD.cs
  using System;
  using Microsoft.Data.Sqlite;

  public static class MetodosCRUD
  {
      // =====================================================
      // 1. CREATE (Maria)
      // =====================================================

      //---a. Cadastrar Vaga---

      public static void CadastrarVaga(SqliteConnection conexao, int idVaga, string titulo, string empresa, decimal salario)
      {
          string query = @"INSERT INTO VAGAS (ID_VAGA, TITULO, EMPRESA, SALARIO)
                          VALUES (@idVaga, @titulo, @empresa, @salario);";

          using (var comando = new SqliteCommand(query, conexao))
          {
              comando.Parameters.AddWithValue("@idVaga", idVaga);
              comando.Parameters.AddWithValue("@titulo", titulo);
              comando.Parameters.AddWithValue("@empresa", empresa);
              comando.Parameters.AddWithValue("@salario", salario);
              comando.ExecuteNonQuery();
          }

          Console.WriteLine($"[OK] Vaga cadastrada: {titulo} - {empresa} (R$ {salario})");
      }

      //---b. Cadastrar Candidata---

      public static void CadastrarCandidata(SqliteConnection conexao, int idCandidata, string nome, string email)
      {
          string query = @"INSERT INTO CANDIDATAS (ID_CANDIDATA, NOME, EMAIL)
                          VALUES (@idCandidata, @nome, @email);";

          using (var comando = new SqliteCommand(query, conexao))
          {
              comando.Parameters.AddWithValue("@idCandidata", idCandidata);
              comando.Parameters.AddWithValue("@nome", nome);
              comando.Parameters.AddWithValue("@email", email);
              comando.ExecuteNonQuery();
          }

          Console.WriteLine($"[OK] Candidata cadastrada: {nome} ({email})");
      }

      //---c. Enviar Candidatura---

      public static void EnviarCandidatura(SqliteConnection conexao, int idCandidatura, int idVaga, int idCandidata)
      {
          string query = @"INSERT INTO CANDIDATURAS (ID_CANDIDATURA, DATA_ENVIO, ID_VAGA, ID_CANDIDATA)
                          VALUES (@idCandidatura, @dataEnvio, @idVaga, @idCandidata);";

          using (var comando = new SqliteCommand(query, conexao))
          {
              comando.Parameters.AddWithValue("@idCandidatura", idCandidatura);
              comando.Parameters.AddWithValue("@dataEnvio", DateTime.Now.ToString("yyyy-MM-dd HH:mm:ss"));
              comando.Parameters.AddWithValue("@idVaga", idVaga);
              comando.Parameters.AddWithValue("@idCandidata", idCandidata);
              comando.ExecuteNonQuery();
          }

          Console.WriteLine($"[OK] Candidatura enviada: Inscrição {idCandidatura} (Vaga {idVaga} / Candidata {idCandidata})");
      }

      // =====================================================
      // 2. READ - Desafio do Double JOIN - Silvia
      // =====================================================
      //--- Consultar candidaturas ---
      public static void ConsultarCandidaturas(SqliteConnection conexao)
      {
          string query = @"SELECT CANDIDATAS.NOME, CANDIDATAS.EMAIL, VAGAS.TITULO, VAGAS.EMPRESA FROM CANDIDATURAS
                            INNER JOIN CANDIDATAS ON CANDIDATURAS.ID_CANDIDATA = CANDIDATAS.ID_CANDIDATA
                            INNER JOIN VAGAS ON CANDIDATURAS.ID_VAGA = VAGAS.ID_VAGA;";
          using(var comando = new SqliteCommand(query, conexao))
          using(var leitor = comando.ExecuteReader())
          {
              Console.WriteLine("\n--- CANDIDATURAS ---");

              while(leitor.Read())
              {
                  Console.WriteLine($"Candidata: {leitor["NOME"]} | " +
                                    $"E-mail: {leitor["EMAIL"]} | " +
                                    $"Vaga: {leitor["TITULO"]} | " +
                                    $"Empresa: {leitor["EMPRESA"]}"
                  );
              }
          }
      }

     // =====================================================
    // 3. UPDATE - Anna
    // =====================================================
    //--- Atualizar Salário da Vaga ---

    public static void AtualizarSalarioVaga(SqliteConnection conexao, int idVaga, decimal novoSalario)
    {
        string query = @"UPDATE VAGAS
                         SET SALARIO = @novoSalario
                         WHERE ID_VAGA = @idVaga;";

        using (var comando = new SqliteCommand(query, conexao))
        {
            comando.Parameters.AddWithValue("@novoSalario", novoSalario);
            comando.Parameters.AddWithValue("@idVaga", idVaga);

            int linhasAfetadas = comando.ExecuteNonQuery();

            if (linhasAfetadas > 0)
            {
                Console.WriteLine($"[OK] Salário da vaga {idVaga} atualizado para R$ {novoSalario}.");
            }
            else
            {
                Console.WriteLine($"[ALERTA] Vaga {idVaga} não encontrada para atualização.");
            }
        }
    }

      // =====================================================
      // DELETE - Kênia
      // =====================================================
      //--- Cancelar Candidatura ---

      public static void CancelarCandidatura(SqliteConnection conexao, int idCandidatura)
      {
          string query = @"DELETE FROM CANDIDATURAS
                          WHERE ID_CANDIDATURA = @idCandidatura;";

          using (var comando = new SqliteCommand(query, conexao))
          {
              comando.Parameters.AddWithValue("@idCandidatura", idCandidatura);

              int linhasAfetadas = comando.ExecuteNonQuery();

              if (linhasAfetadas > 0)
              {
                  Console.WriteLine($"[OK] Candidatura {idCandidatura} cancelada com sucesso.");
              }
              else
              {
                  Console.WriteLine($"[ALERTA] Candidatura {idCandidatura} não encontrada no banco.");
              }
          }
      }
  }


Writing VagasTechApp/MetodosCRUD.cs


In [ ]:
!dotnet build VagasTechApp/

=  Determining projects to restore...
  All projects are up-to-date for restore.
  VagasTechApp -> /content/VagasTechApp/bin/Debug/net8.0/VagasTechApp.dll

Build succeeded.
    0 Warning(s)
    0 Error(s)

Time Elapsed 00:00:10.05


In [ ]:
%%writefile VagasTechApp/Program.cs
// =================================
// PROGRAM - Maria
// =================================


using System;
using Microsoft.Data.Sqlite;

namespace VagasTechApp
{
    class Program
    {
        static void Main(string[] args)
        {
            string connectionString = "Data Source=/content/vagastech.db";

            using (var conexao = new SqliteConnection(connectionString))
            {
                conexao.Open();
                Console.WriteLine("Conexão com vagastech.db aberta com sucesso.\n");

                // 1. Cadastrar duas vagas
                MetodosCRUD.CadastrarVaga(conexao, 1, "Engenheira de Dados", "Empresa Alfa", 9000m);
                MetodosCRUD.CadastrarVaga(conexao, 2, "Analista de BI", "Empresa Ômega", 7500m);

                // 2. Cadastrar a candidata Mariana Souza
                MetodosCRUD.CadastrarCandidata(conexao, 1, "Mariana Souza", "mariana.souza@email.com");

                // 3. Candidatar Mariana nas duas vagas (inscrições 901 e 902)
                MetodosCRUD.EnviarCandidatura(conexao, 901, 1, 1);
                MetodosCRUD.EnviarCandidatura(conexao, 902, 2, 1);

                // 4. Consultar a lista de candidaturas na tela
                MetodosCRUD.ConsultarCandidaturas(conexao);

                // 5. Atualizar o salário da vaga de Engenheira de Dados para R$ 9.500
                MetodosCRUD.AtualizarSalarioVaga(conexao, 1, 9500m);

                // 6. Cancelar a candidatura de Mariana para Analista de BI (inscrição 902)
                MetodosCRUD.CancelarCandidatura(conexao, 902);

                Console.WriteLine("✅ Lista final das candidatas ativas.");
                // 7. Consultar a lista final de candidaturas ativas
                MetodosCRUD.ConsultarCandidaturas(conexao);

                conexao.Close();
                Console.WriteLine("Pipeline finalizado. Conexão encerrada.");
            }
        }
    }
}

Overwriting VagasTechApp/Program.cs


In [ ]:
import os
import sqlite3

caminho = "/content/vagastech.db"

print("Arquivo existe?", os.path.exists(caminho))

if os.path.exists(caminho):
    print("Tamanho do arquivo:", os.path.getsize(caminho), "bytes")

    conexao = sqlite3.connect(caminho)
    cursor = conexao.cursor()

    cursor.execute("""
        SELECT name
        FROM sqlite_master
        WHERE type = 'table'
        ORDER BY name;
    """)

    tabelas = cursor.fetchall()

    print("Tabelas encontradas:")
    print(tabelas)

    conexao.close()

Arquivo existe? True
Tamanho do arquivo: 16384 bytes
Tabelas encontradas:
[('CANDIDATAS',), ('CANDIDATURAS',), ('VAGAS',)]


In [ ]:
!dotnet run --project VagasTechApp/VagasTechApp.csproj

==Conexão com vagastech.db aberta com sucesso.

[OK] Vaga cadastrada: Engenheira de Dados - Empresa Alfa (R$ 9000)
[OK] Vaga cadastrada: Analista de BI - Empresa Ômega (R$ 7500)
[OK] Candidata cadastrada: Mariana Souza (mariana.souza@email.com)
[OK] Candidatura enviada: Inscrição 901 (Vaga 1 / Candidata 1)
[OK] Candidatura enviada: Inscrição 902 (Vaga 2 / Candidata 1)

--- CANDIDATURAS ---
Candidata: Mariana Souza | E-mail: mariana.souza@email.com | Vaga: Engenheira de Dados | Empresa: Empresa Alfa
Candidata: Mariana Souza | E-mail: mariana.souza@email.com | Vaga: Analista de BI | Empresa: Empresa Ômega
[OK] Salário da vaga 1 atualizado para R$ 9500.
[OK] Candidatura 902 cancelada com sucesso.
✅ Lista final das candidatas ativas.

--- CANDIDATURAS ---
Candidata: Mariana Souza | E-mail: mariana.souza@email.com | Vaga: Engenheira de Dados | Empresa: Empresa Alfa
Pipeline finalizado. Conexão encerrada.
=